# Loading Libraries and Packages

In [45]:
import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F


# Setting the random seed for reproducibility
np.random.seed(42)

import warnings
warnings.filterwarnings("ignore")

if torch.backends.mps.is_available():
    print(f"MPS available. \nUsing MPS as runtime.")
    device = torch.device("mps")
    
elif torch.cuda.is_available():
    print(f"CUDA available. \nUsing CUDA as runtime.")
    device = torch.device("cuda")

else:
    print(f"No GPU available. \nUsing CPU runtime.")
    device = torch.device('cpu')

MPS available. 
Using MPS as runtime.


# Phase 4: Self-Supervised Learning via SimCLR (PlantVillage)

## 4.11 InfoNCE Loss

In [7]:
#########  DATASET CLASS  #########

class PlantVillageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.labels_to_index = {}
        self.index_to_labels = {}
        self.samples = []
        self.labels = []   # needed for stratified split

        self.valid_extensions = (".jpg",".jpeg", ".png")
        
        # List of class names (folders in the root directory)
        class_names = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))])

        for labels_index,class_name in enumerate(class_names):
            class_dir = os.path.join(root_dir, class_name)
            self.labels_to_index[class_name] = labels_index
            self.index_to_labels[labels_index] = class_name

            for file_name in os.listdir(class_dir):
                if not file_name.lower().endswith(self.valid_extensions):
                    continue

                path = os.path.join(class_dir, file_name)
                if not os.path.isfile(path):
                    continue
                    
                self.samples.append((path, labels_index))
                self.labels.append(labels_index)
            
    def __len__(self):
        return len(self.samples)

    def num_classes(self):
        return len(self.labels_to_index)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
#########  STRATIFIED SAMPLING  #########

def stratified_split(dataset, random_state=42):
    indices = list(range(len(dataset)))
    labels = dataset.labels

    # Step 1: Train (70) vs Temp (30)
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.3,
        stratify=labels,
        random_state=random_state
    )

    # Labels for temp split
    temp_labels = [labels[i] for i in temp_idx]

    # Step 2: Temp → Val (15) + Test (15)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        stratify=temp_labels,
        random_state=random_state
    )

    return train_idx, val_idx, test_idx

In [40]:
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


root_dir= '../../Assignment 2/plantvillage dataset/color'
train_batch_size = 128
test_batch_size = 64
dataset = PlantVillageDataset(root_dir, transform=None)

train_idx, val_idx, test_idx = stratified_split(dataset)
hundred_test_idx = np.random.choice(test_idx, size = 100, replace= False)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

In [22]:
import torchvision.transforms as transforms

contrastive_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
])

In [23]:
class ContrastiveDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        x1 = self.transform(img)
        x2 = self.transform(img)
        return x1, x2

In [24]:
contrastive_train_dataset = ContrastiveDataset(train_dataset, contrastive_transform)
contrastive_val_dataset   = ContrastiveDataset(val_dataset, contrastive_transform)

train_loader = DataLoader(contrastive_train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(contrastive_val_dataset, batch_size=64, shuffle=False)

In [30]:
class SimCLR(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        
        base_model = models.resnet18(pretrained=False)
        self.encoder = nn.Sequential(*list(base_model.children())[:-1])  # remove fc
        
        self.projector = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return z

In [31]:
def info_nce_loss(z1, z2, temperature=0.5):
    batch_size = z1.size(0)
    
    z = torch.cat([z1, z2], dim=0)  # [2B, D]
    sim_matrix = torch.matmul(z, z.T)  # cosine similarity since normalized
    
    # remove self-similarity
    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)

    sim_matrix = sim_matrix / temperature

    # positive pairs
    positives = torch.cat([
        torch.diag(sim_matrix, batch_size),
        torch.diag(sim_matrix, -batch_size)
    ], dim=0)

    labels = torch.zeros(2 * batch_size).long().to(z.device)

    logits = torch.cat([positives.unsqueeze(1), sim_matrix], dim=1)
    
    loss = F.cross_entropy(logits, labels)
    return loss

In [33]:
model = SimCLR().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

epochs = 10
log_interval = 50

save_dir = "../Models"
model_name = "4_11_1.pth"
os.makedirs(save_dir, exist_ok=True)

best_val_loss = float("inf")

for epoch in range(epochs):
    # ===== TRAIN =====
    model.train()
    total_train_loss = 0

    for batch_idx, (x1, x2) in enumerate(train_loader):
        x1, x2 = x1.to(device), x2.to(device)

        z1 = model(x1)
        z2 = model(x2)

        loss = info_nce_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        # batch logging
        if (batch_idx + 1) % log_interval == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] | "
                f"Batch [{batch_idx+1}/{len(train_loader)}] | "
                f"Train Loss: {loss.item():.4f}"
            )

    avg_train_loss = total_train_loss / len(train_loader)

    # ===== VALIDATION =====
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for x1, x2 in val_loader:
            x1, x2 = x1.to(device), x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = info_nce_loss(z1, z2)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    # ===== PRINT =====
    print(
        f"\nEpoch [{epoch+1}/{epochs}] Summary:\n"
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}\n"
    )

    # ===== SAVE LAST CHECKPOINT =====
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
    }, os.path.join(save_dir, model_name))

    # ===== SAVE BEST MODEL =====
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": avg_val_loss,
        }, os.path.join(save_dir, model_name))

        print("Best model updated.\n")

Epoch [1/10] | Batch [50/297] | Train Loss: 4.8162
Epoch [1/10] | Batch [100/297] | Train Loss: 4.5538
Epoch [1/10] | Batch [150/297] | Train Loss: 4.5815
Epoch [1/10] | Batch [200/297] | Train Loss: 4.4915
Epoch [1/10] | Batch [250/297] | Train Loss: 4.4762

Epoch [1/10] Summary:
Train Loss: 4.5988 | Val Loss: 3.9201

Best model updated.

Epoch [2/10] | Batch [50/297] | Train Loss: 4.3534
Epoch [2/10] | Batch [100/297] | Train Loss: 4.2982
Epoch [2/10] | Batch [150/297] | Train Loss: 4.2265
Epoch [2/10] | Batch [200/297] | Train Loss: 4.1990
Epoch [2/10] | Batch [250/297] | Train Loss: 4.2493

Epoch [2/10] Summary:
Train Loss: 4.2673 | Val Loss: 3.5754

Best model updated.

Epoch [3/10] | Batch [50/297] | Train Loss: 4.1600
Epoch [3/10] | Batch [100/297] | Train Loss: 4.1162
Epoch [3/10] | Batch [150/297] | Train Loss: 4.1080
Epoch [3/10] | Batch [200/297] | Train Loss: 4.0403
Epoch [3/10] | Batch [250/297] | Train Loss: 4.1164

Epoch [3/10] Summary:
Train Loss: 4.1165 | Val Loss: 3.4

In [34]:
model.eval()
val_loss = 0

with torch.no_grad():
    for x1, x2 in val_loader:
        x1, x2 = x1.to(device), x2.to(device)

        z1 = model(x1)
        z2 = model(x2)

        loss = info_nce_loss(z1, z2)
        val_loss += loss.item()

val_loss /= len(val_loader)

print(f"Final Validation Loss: {val_loss:.4f}")

Final Validation Loss: 3.2273


## 4.12 The Linear Evaluation Protocol

In [46]:
save_dir = "../Models"
model_name = "4_11_1.pth"
model_path = os.path.join(save_dir, model_name)
model = SimCLR().to(device)

checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

encoder = model.encoder  # extract encoder

for param in encoder.parameters():
    param.requires_grad = False

encoder.eval()
print("Model loaded Successfully!")

KeyboardInterrupt: 

In [41]:
num_classes = dataset.num_classes()
classifier = nn.Linear(512, num_classes).to(device)

dataset_sup = PlantVillageDataset(root_dir, transform=base_transform)
train_dataset_sup = Subset(dataset_sup, train_idx)
val_dataset_sup   = Subset(dataset_sup, val_idx)
test_dataset_sup  = Subset(dataset_sup, test_idx)

train_loader_sup = DataLoader(train_dataset_sup, batch_size=128, shuffle=True)
val_loader_sup   = DataLoader(val_dataset_sup, batch_size=64, shuffle=False)
test_loader_sup  = DataLoader(test_dataset_sup, batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=5e-4)

In [43]:
save_dir = "../Models"
model_name = "4_11_1_linear.pth"

epochs = 20
log_interval = 50
patience = 5   # early stopping patience

save_path = os.path.join(save_dir, model_name)

best_val_acc = 0
epochs_no_improve = 0

for epoch in range(epochs):
    # ===== TRAIN =====
    classifier.train()
    total_loss = 0

    for batch_idx, (images, labels) in enumerate(train_loader_sup):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            features = encoder(images)
            features = features.view(features.size(0), -1)

        outputs = classifier(features)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # batch-wise logging
        if (batch_idx + 1) % log_interval == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] | "
                f"Batch [{batch_idx+1}/{len(train_loader_sup)}] | "
                f"Train Loss: {loss.item():.4f}"
            )

    avg_train_loss = total_loss / len(train_loader_sup)

    # ===== VALIDATION =====
    classifier.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader_sup:
            images, labels = images.to(device), labels.to(device)

            features = encoder(images)
            features = features.view(features.size(0), -1)

            outputs = classifier(features)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total

    # ===== PRINT =====
    print(
        f"\nEpoch [{epoch+1}/{epochs}] Summary:\n"
        f"Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.2f}%\n"
    )

    # ===== CHECKPOINT (BEST) =====
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0

        torch.save({
            "epoch": epoch + 1,
            "classifier_state_dict": classifier.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
        }, save_path)

        print("Best linear model updated.\n")

    else:
        epochs_no_improve += 1

    # ===== EARLY STOPPING =====
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

Epoch [1/20] | Batch [50/297] | Train Loss: 0.5169
Epoch [1/20] | Batch [100/297] | Train Loss: 0.4918
Epoch [1/20] | Batch [150/297] | Train Loss: 0.4353
Epoch [1/20] | Batch [200/297] | Train Loss: 0.3244
Epoch [1/20] | Batch [250/297] | Train Loss: 0.3086

Epoch [1/20] Summary:
Train Loss: 0.4416 | Val Acc: 88.88%

Best linear model updated.

Epoch [2/20] | Batch [50/297] | Train Loss: 0.4212
Epoch [2/20] | Batch [100/297] | Train Loss: 0.3676
Epoch [2/20] | Batch [150/297] | Train Loss: 0.2932
Epoch [2/20] | Batch [200/297] | Train Loss: 0.4297
Epoch [2/20] | Batch [250/297] | Train Loss: 0.3240

Epoch [2/20] Summary:
Train Loss: 0.3667 | Val Acc: 90.08%

Best linear model updated.

Epoch [3/20] | Batch [50/297] | Train Loss: 0.2779
Epoch [3/20] | Batch [100/297] | Train Loss: 0.3280
Epoch [3/20] | Batch [150/297] | Train Loss: 0.2748
Epoch [3/20] | Batch [200/297] | Train Loss: 0.2288
Epoch [3/20] | Batch [250/297] | Train Loss: 0.3894

Epoch [3/20] Summary:
Train Loss: 0.3249 | V

In [44]:
classifier.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader_sup:
        images, labels = images.to(device), labels.to(device)

        features = encoder(images)
        features = features.view(features.size(0), -1)

        outputs = classifier(features)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = 100 * correct / total

print(f"Test Accuracy (Linear Eval): {test_acc:.2f}%")

Test Accuracy (Linear Eval): 94.55%
